# ModelForge Lite — Phase 4: LoRA Fine-Tuning

This notebook fine-tunes the base model on `train.csv` using LoRA (Low-Rank Adaptation) — a parameter-efficient technique that trains a small set of additional weights instead of the whole model. This produces Variant 3 (fine-tuned, no RAG). Variant 4 (fine-tuned + RAG) reuses this same adapter in Phase 5.

Run in Colab with the T4 GPU enabled.

In [ ]:
!pip install -q transformers accelerate datasets huggingface_hub pandas peft bitsandbytes

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Pull the training split from Phase 1

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

HF_USERNAME = "YOUR_HF_USERNAME"
dataset_repo = f"{HF_USERNAME}/modelforge-lite-support-data"

train_path = hf_hub_download(repo_id=dataset_repo, filename="train.csv", repo_type="dataset")
train_df = pd.read_csv(train_path)
print(f"Training rows: {len(train_df)}")
train_df.head()

## 2. Format the data for instruction fine-tuning

Each row becomes a (system prompt + question) -> (answer) example, in the same chat format the model expects. Note this uses NO retrieval and NO reference context — the goal here is for the model to internalize the support-agent style and domain knowledge directly into its weights.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SYSTEM_PROMPT = (
    "You are a helpful customer support assistant. "
    "Answer the customer's question clearly and concisely."
)

def format_example(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

hf_dataset = Dataset.from_pandas(train_df)
hf_dataset = hf_dataset.map(format_example)
print(hf_dataset[0]["text"])

## 3. Tokenize

In [ ]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = hf_dataset.map(tokenize_fn, remove_columns=hf_dataset.column_names)
print(tokenized_dataset)

## 4. Load the base model and attach a LoRA adapter

LoRA freezes the original model weights entirely and injects small trainable "adapter" matrices into specific layers (here, the attention projection layers). Only those adapter weights get updated during training — typically under 1% of the model's total parameters. That's what makes this feasible on a free GPU in well under an hour, and why the resulting checkpoint is tens of MB instead of gigabytes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                 # rank of the adapter matrices — higher = more capacity, more params
    lora_alpha=16,        # scaling factor for adapter updates
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # which layers get adapters
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 5. Train

With a 0.5B model and a few hundred training rows, this should take somewhere in the range of 10-30 minutes on a free T4. Watch the loss column — it should trend downward.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora-checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

## 6. Save and push the LoRA adapter to Hugging Face Hub

Only the small adapter gets uploaded — not the full base model. Loading it later just means: load the base model, then attach this adapter on top.

In [ ]:
adapter_repo = f"{HF_USERNAME}/modelforge-lite-lora-adapter"

model.push_to_hub(adapter_repo)
tokenizer.push_to_hub(adapter_repo)
print(f"Adapter pushed to: https://huggingface.co/{adapter_repo}")

## 7. Quick test — compare a fine-tuned answer to the base model's

Run a couple of training-domain questions through the fine-tuned model directly (no RAG yet) to sanity check the training worked.

In [ ]:
def generate_finetuned_answer(question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print(generate_finetuned_answer("How can I get a refund for my order?"))

## Done — Phase 4 checklist

- [ ] Training data formatted into chat-style examples
- [ ] LoRA adapter attached to the base model (only ~1% of params trainable)
- [ ] Training completed, loss trending down
- [ ] Adapter pushed to Hugging Face Hub
- [ ] Quick sanity-check answer looks domain-appropriate

## What to be able to explain on the call
- **What is LoRA, in plain terms?** Instead of updating billions of parameters, you freeze the whole base model and insert small trainable matrices into a few key layers. It's much cheaper to train and produces a tiny, portable adapter file instead of a full new model copy.
- **Why r=8 and those target_modules?** These are the attention layers that most influence how the model weighs and uses information — a common, effective choice for lightweight adaptation. Rank 8 is a modest capacity that suits a small dataset like this one; a much higher rank on this little data would risk overfitting.
- **What does this variant lack, compared to fine-tuned + RAG?** It has learned the *style and general domain patterns* from training, but it can only draw on what was in the (relatively small) training set. It can't look up new or updated information — it can only reproduce patterns it saw during training. That's exactly what Phase 5 tests.

Next: Phase 5 — Fine-tuned model + RAG combined, and running both new variants through eval.csv.